# Titanic data pipeline

## Eindresultaat

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVC
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

# data inladen
titanic_df = pd.read_csv("data/titanic_dataset.csv")

# Features selecteren, voorspeld item aanduiden (overleefd of niet)
X = titanic_df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]
y = titanic_df['Survived']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Preprocessing voor numerieke features
numeric_features = ['Age', 'SibSp', 'Parch', 'Fare']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), ## dit vangt gemiste waardes op
    ('scaler', StandardScaler())
])

# Preprocessing voor categorische features
categorical_features = ['Pclass', 'Sex', 'Embarked']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# combineer deze transformators door aan te duiden welke waar moet werken
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Encodeer de target variabele
label_encoder = LabelEncoder() #label omzetten in een numerieke waarde, in dit geval was dit eigenlijk al numeriek
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

# SVM Classifier
svm_clf = SVC()

# pipeline maken met alle stukken
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', svm_clf)
])
vsm_trained = pipeline.named_steps['classifier']

pipeline.fit(X_train, y_train)

# Grid search parameters
param_grid = {
    'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100],  # SVM regularization parameter
    'classifier__kernel': ['linear', 'rbf', 'poly'],  # Kernel types
}

# grid search met cross-validation
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')

# Fitten (trainen van data)
grid_search.fit(X_test, y_test)

# Print beste parameters
print("Best Parameters: ", grid_search.best_params_)
print("Best Cross-Validation Score: {:.2f}".format(grid_search.best_score_))

# Evaluatie
y_predicted = pipeline.predict(X_train)
best_model = grid_search.best_estimator_
test_score = best_model.score(X_test, y_test)
print("Test Set Accuracy: {:.2f}".format(test_score))

## Stap per stap opgebouwd

Data inladen

In [ ]:
# data inladen
titanic_df = pd.read_csv("data/titanic_dataset.csv")
titanic_df.head()

Beetje rondkijken welke data je in huis hebt. Typische commando's: df.head() of df.tail(), df.dtypes, df.describe()

In [ ]:
titanic_df.dtypes

In [ ]:
titanic_df.describe()

Let hierboven goed op, de afwijkende count van Age geeft aan dat er Null values tussenzitten. Dit gaan we opvangen.

In [ ]:
titanic_df.count()

In dit geval is PassengerId, Ticket, Cabin geen waardevolle informatie. Ik ga ze niet meenemen in het model. Je kan dit doen door expliciet de andere kolommen op te lijsten of gebruik te maken van de .drop functionaliteit (zie documentatie pandas voor meer details)

In [ ]:
X = titanic_df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]
y = titanic_df['Survived']


Ik splits mijn data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Ik behandel de numerieke features door Null values expliciet op te vangen met een zinvolle strategie, en daarna gebruik ik een standardscaler.

In [ ]:
numeric_features = ['Age', 'SibSp', 'Parch', 'Fare']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), ## dit vangt gemiste waardes op
    ('scaler', StandardScaler())
])
print(numeric_transformer)
numeric_transformer

De categorische features ga ik OneHot encoden:

In [ ]:
# Preprocessing voor categorische features
categorical_features = ['Pclass', 'Sex', 'Embarked']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
categorical_transformer

Ook de labels ga ik door een simpele encoder jagen.

In [ ]:
label_encoder = LabelEncoder() #label omzetten in een numerieke waarde, in dit geval was dit eigenlijk al numeriek
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

De volledige pipeline ziet er dan zo uit:

In [ ]:
# SVM Classifier
svm_clf = SVC()

# pipeline maken met alle stukken
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', svm_clf)
])
svm_algo = pipeline.named_steps['classifier']
pipeline

In [ ]:
svm_algo

Nu kan de data worden gefit:

In [ ]:
pipeline.fit(X_train, y_train)

Nu kan ik nog grid Search doen om de beste parameters voor het model te bepalen:

In [ ]:
# Grid search parameters
param_grid = {
    'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100],  # SVM regularization parameter
    'classifier__kernel': ['linear', 'rbf', 'poly'],  # Kernel types
}

# grid search met cross-validation
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')

# Fitten (trainen van data)
grid_search.fit(X_test, y_test)

De beste parameters voor dit model zijn

In [ ]:
print("Best Parameters: ", grid_search.best_params_)
print("Best Cross-Validation Score: {:.2f}".format(grid_search.best_score_))

En hiervoor heb je deze score:

In [ ]:
# Evaluatie
y_predicted = pipeline.predict(X_train)
best_model = grid_search.best_estimator_
test_score = best_model.score(X_test, y_test)
print("Test Set Accuracy: {:.2f}".format(test_score))